<a href="https://colab.research.google.com/github/jasminealyaa1212/DIP-final-project/blob/main/coba%20dulu%20deh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% CELL 1 - Config
import os
import urllib.request
from collections import deque

import cv2
import numpy as np

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
except ImportError:
    pass

BASE = '/content/drive/MyDrive/semester 3/Digital Image Processing/final project'
VIDEO_PATH = f'{BASE}/video 5.mp4'
OUTPUT_PATH = f'{BASE}/annotated_output.mp4'
SNAP_DIR = f'{BASE}/violations'

# Allowed traffic direction as a vector (dx, dy); image y grows DOWNWARD.
# (0, 1) = down the screen, (0, -1) = up, (-2, -1) = toward upper-left, etc.
ALLOWED_VECTOR = (0, 1)

# Optional per-lane directions (coordinates in the W x H processing frame below).
# LANES = [{'name': 'L', 'vector': (0, 1),  'polygon': [(50, 350), (250, 350), (275, 100), (150, 100)]},
#          {'name': 'R', 'vector': (0, -1), 'polygon': [(260, 350), (500, 350), (400, 100), (285, 100)]}]
LANES = []

W, H = 640, 360              # processing size (raise to 960x540 if vehicles are tiny)
FRAME_SKIP = 3               # analyse every 3rd frame
WARMUP = 15                  # processed frames used to learn the background
MIN_BLOB_AREA = 700          # smallest moving blob (px^2) treated as a vehicle
USE_BLOB_FALLBACK = True     # keep moving blobs even if no cascade recognises them
HIST_LEN, MIN_HIST, MAX_MISSED = 10, 5, 5
MIN_TRAVEL_RATIO = 0.2       # net travel must exceed this * box height
COS_THR = 0.5                # direction must be within 60 deg of the road axis
VOTES_TO_FLAG = 3            # consecutive wrong-way verdicts -> violation

CASCADES = {
    'car': 'https://raw.githubusercontent.com/andrewssobral/vehicle_detection_haarcascades/master/cars.xml',
    'two_wheeler': 'https://raw.githubusercontent.com/AdityaPai2398/Vehicle-And-Pedestrian-Detection-Using-Haar-Cascades/master/Main%20Project/Main%20Project/Bike%20Detection/two_wheeler.xml',
    'bus': 'https://raw.githubusercontent.com/AdityaPai2398/Vehicle-And-Pedestrian-Detection-Using-Haar-Cascades/master/Main%20Project/Main%20Project/Bus%20Detection/Bus_front.xml',
}


# %% CELL 2 - Pipeline
def load_cascades():
    out = {}
    for name, url in CASCADES.items():
        f = f'{name}.xml'
        try:
            if not os.path.exists(f):
                urllib.request.urlretrieve(url, f)
            c = cv2.CascadeClassifier(f)
            if not c.empty():
                out[name] = c
        except Exception as e:
            print(f'Could not load {name}: {e}')
    print('Cascades loaded:', list(out))
    return out


def detect(gray, fg, cascades):
    """Moving blobs (background subtraction) classified by Haar cascades on small crops."""
    dets = []
    contours, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contours:
        if cv2.contourArea(c) < MIN_BLOB_AREA:
            continue
        x, y, w, h = cv2.boundingRect(c)
        roi = gray[max(0, y - 8):y + h + 8, max(0, x - 8):x + w + 8]   # Haar only runs here
        label = next((n for n, cas in cascades.items()
                      if len(cas.detectMultiScale(roi, 1.1, 3, minSize=(16, 16)))), None)
        if label or USE_BLOB_FALLBACK:
            dets.append((x, y, w, h, label or 'vehicle'))
    return dets


class Track:
    def __init__(self, tid, box, label):
        self.id, self.label = tid, label
        self.hist = deque(maxlen=HIST_LEN)
        self.missed = self.votes = 0
        self.flagged = self.saved = False
        self.add(box)

    def add(self, box):
        self.box = box
        x, y, w, h = box
        self.hist.append((x + w / 2, y + h / 2))

    def judge(self, vec):
        """'wrong', 'ok' or None, from the least-squares motion direction vs. road vector."""
        if len(self.hist) < MIN_HIST:
            return None
        pts = np.array(self.hist, dtype=float)
        disp = np.polyfit(np.arange(len(pts)), pts, 1)[0] * (len(pts) - 1)   # (dx, dy)
        dist = np.hypot(*disp)
        if dist < MIN_TRAVEL_RATIO * self.box[3]:
            return None
        cos = disp @ np.array(vec, float) / (dist * np.hypot(*vec))
        return 'wrong' if cos < -COS_THR else 'ok' if cos > COS_THR else None


def update_tracks(tracks, dets, next_id):
    """Greedy nearest-centroid matching of detections to existing tracks."""
    items = list(tracks.values())
    pairs = sorted((np.hypot(t.hist[-1][0] - (x + w / 2), t.hist[-1][1] - (y + h / 2)), i, j)
                   for i, t in enumerate(items) for j, (x, y, w, h, _) in enumerate(dets)
                   if np.hypot(t.hist[-1][0] - (x + w / 2), t.hist[-1][1] - (y + h / 2)) < max(30, 1.5 * h))
    used_t, used_d = set(), set()
    for _, i, j in pairs:
        if i in used_t or j in used_d:
            continue
        items[i].add(dets[j][:4]); items[i].label = dets[j][4]; items[i].missed = 0
        used_t.add(i); used_d.add(j)
    for i, t in enumerate(items):
        if i not in used_t:
            t.missed += 1
    for j, d in enumerate(dets):
        if j not in used_d:
            tracks[next_id] = Track(next_id, d[:4], d[4]); next_id += 1
    for k in [k for k, t in tracks.items() if t.missed > MAX_MISSED]:
        del tracks[k]
    return next_id


def vector_at(pt):
    """Allowed vector at point pt (None if outside every lane)."""
    if not LANES:
        return ALLOWED_VECTOR
    for lane in LANES:
        if cv2.pointPolygonTest(np.array(lane['polygon'], np.int32), pt, False) >= 0:
            return lane['vector']
    return None


# %% CELL 3 - Run
def run():
    cascades = load_cascades()
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise FileNotFoundError(f'Cannot open {VIDEO_PATH}')
    os.makedirs(SNAP_DIR, exist_ok=True)
    fps = (cap.get(cv2.CAP_PROP_FPS) or 25) / FRAME_SKIP
    out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))

    mog = cv2.createBackgroundSubtractorMOG2(history=150, varThreshold=32, detectShadows=True)
    clahe = cv2.createCLAHE(2.0, (8, 8))
    k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

    tracks, next_id, violators, seen = {}, 0, set(), set()
    n = processed = 0
    while cap.grab():                      # grab() skips decoding work for dropped frames
        n += 1
        if n % FRAME_SKIP:
            continue
        ok, frame = cap.retrieve()
        if not ok:
            break
        frame = cv2.resize(frame, (W, H), interpolation=cv2.INTER_AREA)
        gray = clahe.apply(cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0))

        fg = mog.apply(frame)
        processed += 1
        if processed <= WARMUP:
            out.write(frame)
            continue
        fg = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)[1]          # drop shadows (127)
        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, k_open)               # remove speckle
        fg = cv2.dilate(cv2.morphologyEx(fg, cv2.MORPH_CLOSE, k_close), k_open)  # fill holes

        next_id = update_tracks(tracks, detect(gray, fg, cascades), next_id)

        for t in tracks.values():
            if t.missed:
                continue
            seen.add(t.id)
            vec = vector_at(t.hist[-1])
            verdict = t.judge(vec) if vec else None
            t.votes = t.votes + 1 if verdict == 'wrong' else 0 if verdict == 'ok' else t.votes
            if t.votes >= VOTES_TO_FLAG:
                t.flagged = True
                violators.add(t.id)

            x, y, w, h = t.box
            color = (0, 0, 255) if t.flagged else (0, 255, 0)
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, f"#{t.id} {'WRONG WAY' if t.flagged else 'OK'} [{t.label}]",
                        (x, max(12, y - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            cv2.polylines(frame, [np.array(t.hist, np.int32).reshape(-1, 1, 2)], False, color, 1)
            if t.flagged and not t.saved:
                cv2.imwrite(f'{SNAP_DIR}/violation_id{t.id}_f{n}.jpg', frame)
                t.saved = True

        for lane in LANES:
            cv2.polylines(frame, [np.array(lane['polygon'], np.int32)], True, (255, 200, 0), 1)
        cv2.putText(frame, f'Vehicles: {len(seen)}  Wrong-way: {len(violators)}', (8, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 200, 255), 2)
        out.write(frame)

    cap.release(); out.release()
    print(f'Done: {processed} frames | vehicles: {len(seen)} | wrong-way: {len(violators)}')
    print(f'Video: {OUTPUT_PATH}')


# %% CELL 4 (optional) - save a gridded reference frame to read lane coordinates
def save_reference_frame(at=60):
    cap = cv2.VideoCapture(VIDEO_PATH); cap.set(cv2.CAP_PROP_POS_FRAMES, at)
    f = cv2.resize(cap.read()[1], (W, H)); cap.release()
    for x in range(0, W, 50):
        cv2.line(f, (x, 0), (x, H), (255, 255, 255), 1); cv2.putText(f, str(x), (x + 2, 10), 0, 0.3, (0, 255, 255), 1)
    for y in range(0, H, 50):
        cv2.line(f, (0, y), (W, y), (255, 255, 255), 1); cv2.putText(f, str(y), (2, y - 2), 0, 0.3, (0, 255, 255), 1)
    cv2.imwrite(f'{BASE}/reference_frame.jpg', f)
# save_reference_frame()


run()

Cascades loaded: ['car', 'two_wheeler', 'bus']
Done: 161 frames | vehicles: 19 | wrong-way: 1
Video: /content/drive/MyDrive/semester 3/Digital Image Processing/final project/annotated_output.mp4
